# Análisis FONTUR Profundo (Parte 1): Eficiencia, Concentración y Poder de Contratación

In [1]:
import pandas as pd
import json
import os
DATA_DIR = '../data'
OUTPUT_DIR = '../src/data'
os.makedirs(OUTPUT_DIR, exist_ok=True)
files = [f for f in os.listdir(DATA_DIR) if f.endswith('.xlsx')]

## 1. Eficiencia Operativa (Presupuesto Macro)

In [2]:
file_presupuesto = [f for f in files if 'Presupuesto ejecutado' in f or '453994' in f][0]
df_presupuesto = pd.read_excel(os.path.join(DATA_DIR, file_presupuesto), sheet_name='1. Presupuesto 2021-2024', header=4)
df_presupuesto.columns = [str(c).replace('\n', ' ').strip() for c in df_presupuesto.columns]
df_presupuesto = df_presupuesto.dropna(subset=['Vigencia'])
df_presupuesto['Eficiencia_Pago_Vs_Aprobado'] = (df_presupuesto['Pagado'] / df_presupuesto['Presupuesto aprobado Vigencia']) * 100
display(df_presupuesto[['Vigencia', 'Presupuesto aprobado Vigencia', 'Contratado', 'Pagado', 'Eficiencia_Pago_Vs_Aprobado']])
with open(os.path.join(OUTPUT_DIR, 'fontur_presupuesto_macro.json'), 'w') as f:
    json.dump(df_presupuesto.fillna(0).to_dict(orient='records'), f, indent=4)

,Vigencia,Presupuesto aprobado Vigencia,Contratado,Pagado,Eficiencia_Pago_Vs_Aprobado
0,2021,285619576028.341187,1.684278e+11,7.407958e+10,25.93645
1,2022,279295532559.796082,1.120571e+11,5.716573e+10,20.467827
2,2023,576215436042.199951,3.689212e+11,1.446475e+11,25.103022
3,2024-10,535087851842.619995,NaN,NaN,NaN
4,Total al cierre,1676218396472.957031,6.494062e+11,2.758928e+11,16.459239


/var/folders/58/pn94jq9s28nc3tbhycyxxfkw0000gn/T/ipykernel_12007/523767639.py:8: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  json.dump(df_presupuesto.fillna(0).to_dict(orient='records'), f, indent=4)


## 2. Índice de Concentración (Modalidades)

In [3]:
file_recursos = [f for f in files if '453995' in f or 'RecursosContratados' in f][0]
df_modalidad = pd.read_excel(os.path.join(DATA_DIR, file_recursos), sheet_name='Modalidad_Contratación', header=1)
df_modalidad.columns = [str(c).replace('\n', ' ').strip() for c in df_modalidad.columns]
df_modalidad = df_modalidad.dropna(how='all')
with open(os.path.join(OUTPUT_DIR, 'fontur_modalidad.json'), 'w') as f:
    json.dump(df_modalidad.fillna(0).to_dict(orient='records'), f, indent=4)

## 3. Minería de Contratos (Muestra granular)

In [4]:
file_aclaracion = [f for f in files if 'Aclaracin' in f or '453996' in f][0]
df_contratos = pd.read_excel(os.path.join(DATA_DIR, file_aclaracion), sheet_name='Contratos', header=0)
df_contratos.columns = [str(c).strip() for c in df_contratos.columns]
res_mod = df_contratos.groupby('MODALIDAD DE CONTRATACIÓN').agg(
    Total_Contratos=('N° CONTRATO', 'count'), Valor_Suma=('VALOR INICIAL Y APORTADOS POR FONTUR', 'sum')
).reset_index()
res_mod['Porcentaje'] = (res_mod['Valor_Suma'] / res_mod['Valor_Suma'].sum()) * 100
display(res_mod.sort_values('Valor_Suma', ascending=False))
with open(os.path.join(OUTPUT_DIR, 'fontur_analisis_profundo.json'), 'w') as f:
    json.dump({'resumen_modalidad': res_mod.fillna(0).to_dict(orient='records')}, f, indent=4)
print("✅ FONTUR 01 Procesado Exitosamente")

,MODALIDAD DE CONTRATACIÓN,Total_Contratos,Valor_Suma,Porcentaje
2,CONTRATACIÓN DIRECTA,939,5.151985e+11,49.268068
6,INVITACIÓN ABIERTA,147,2.776944e+11,26.555718
0,COMPARACIÓN DE COTIZACIONES,66,1.102974e+11,10.547658
4,CONVENIOS,65,9.629889e+10,9.208995
8,ORDEN DE SERVICIO POR COMPRA LOGISTICA,566,3.982165e+10,3.808116
7,INVITACIÓN PRIVADA,15,4.039486e+09,0.386293
3,CONTRATO DE ADHESIÓN,4,1.460883e+09,0.139703
5,INSTRUMENTO DE AGREGACION DE DEMANDA,9,8.805962e+08,0.084211
1,CONTRATACION PARTICIPACION FERIAL,1,1.294334e+07,0.001238


✅ FONTUR 01 Procesado Exitosamente
